# Importando json de informações gerais da API

In [15]:
# Passo 1: Importando JSON completo da API da EPL

import requests

url = 'https://fantasy.premierleague.com/api/bootstrap-static/'

# É recomendável adicionar um User-Agent para evitar bloqueios (erro 403)
headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/100.0.0.0 Safari/537.36'
}

response = requests.get(url, headers=headers)

# Verifica se a requisição foi bem-sucedida
if response.status_code == 200:

    # Convertendo a resposta em json
    data = response.json()
    print("Sucesso! Dados carregados.")
    
    # Exemplo: Acessando os nomes dos jogadores
    # Os jogadores ficam na chave 'elements'
    primeiro_jogador = data['elements'][0]
    print(f"Exemplo de jogador: {primeiro_jogador['second_name']}")
else:
    print(f"Erro ao acessar API: {response.status_code}")

Sucesso! Dados carregados.
Exemplo de jogador: Raya Martín


In [16]:
# Salvando arquivo json com dados

import json
from datetime import datetime

# Nome do arquivo que será criado
agora = datetime.now()
data_formatada = agora.strftime("%d-%m-%Y_%H-%M-%S")
nome_arquivo = f"dados_fpl_{data_formatada}.json"

# Abrindo o arquivo em modo de escrita ('w' de write)
with open(nome_arquivo, 'w', encoding='utf-8') as f:
    # dump() despeja o conteúdo da variável no arquivo
    # indent=4 serve para deixar o arquivo legível para humanos
    # ensure_ascii=False garante que acentos e caracteres especiais fiquem corretos
    json.dump(data, f, indent=4, ensure_ascii=False)

print(f"Arquivo {nome_arquivo} salvo com sucesso!")

Arquivo dados_fpl_14-04-2026_17-15-57.json salvo com sucesso!


# Mapa de chaves (verificação)

In [17]:
# Mapa de chaves do json

import pandas as pd

# Dicionário para armazenar o nome da chave e suas colunas/sub-chaves
mapa_colunas = {}

for chave, valor in data.items():
    if isinstance(valor, list) and len(valor) > 0:
        # Se for uma lista de dicionários (como 'elements')
        if isinstance(valor[0], dict):
            mapa_colunas[chave] = list(valor[0].keys())
        else:
            mapa_colunas[chave] = ["Lista de valores simples"]
            
    elif isinstance(valor, dict):
        # Se for um dicionário único (como 'game_settings')
        mapa_colunas[chave] = list(valor.keys())
    else:
        mapa_colunas[chave] = ["Valor escalar/único"]

# Exibindo os resultados de forma organizada
for chave, colunas in mapa_colunas.items():
    print(f"--- Chave: {chave} ---")
    print(f"Colunas/Campos: {colunas}")
    print("\n")

--- Chave: chips ---
Colunas/Campos: ['id', 'name', 'number', 'start_event', 'stop_event', 'chip_type', 'overrides']


--- Chave: events ---
Colunas/Campos: ['id', 'name', 'deadline_time', 'release_time', 'average_entry_score', 'finished', 'data_checked', 'highest_scoring_entry', 'deadline_time_epoch', 'deadline_time_game_offset', 'highest_score', 'is_previous', 'is_current', 'is_next', 'cup_leagues_created', 'h2h_ko_matches_created', 'can_enter', 'can_manage', 'released', 'ranked_count', 'overrides', 'chip_plays', 'most_selected', 'most_transferred_in', 'top_element', 'top_element_info', 'transfers_made', 'most_captained', 'most_vice_captained']


--- Chave: game_settings ---
Colunas/Campos: ['league_join_private_max', 'league_join_public_max', 'league_max_size_public_classic', 'league_max_size_public_h2h', 'league_max_size_private_h2h', 'league_max_ko_rounds_private_h2h', 'league_prefix_public', 'league_points_h2h_win', 'league_points_h2h_lose', 'league_points_h2h_draw', 'league_ko_f

# Importando json de histórico

In [18]:
import glob

# 1. Busca todos os arquivos que começam com 'dados_fpl_' e terminam com '.json'
padrao = 'historico_fpl_*.json'
arquivos = glob.glob(padrao)

if not arquivos:
    print("Nenhum arquivo encontrado com esse padrão.")
else:
    # 2. Identifica o arquivo com a data de modificação mais recente
    arquivo_mais_recente = max(arquivos, key=os.path.getmtime)
    
    print(f"Abrindo o arquivo mais recente: {arquivo_mais_recente}")

    # 3. Abre e carrega o conteúdo do JSON
    with open(arquivo_mais_recente, 'r', encoding='utf-8') as f:
        data_hist = json.load(f)
    
    print(f"Sucesso!")

Abrindo o arquivo mais recente: historico_fpl_14-04-2026_16-56-40.json
Sucesso!


# Verificando se dados históricos estão atualizados

In [19]:
import time

# --- PASSO 1: Identificar a última rodada finalizada no 'data' ---
# Filtramos as rodadas onde 'finished' é True e pegamos o maior 'id'
rodadas_finalizadas = [e['id'] for e in data['events'] if e['finished'] is True]
ultima_rodada_real = max(rodadas_finalizadas) if rodadas_finalizadas else 0

print(f"Última rodada finalizada na API: {ultima_rodada_real}")

# --- PASSO 2: Verificar o maior round no seu 'data_hist' ---
# Como data_hist é uma lista de dicionários, podemos usar um generator para achar o máximo
maior_round_local = max([item['round'] for item in data_hist]) if data_hist else 0

print(f"Maior rodada no seu arquivo local: {maior_round_local}")

# --- PASSO 3: Condicional para atualização ---
if maior_round_local < ultima_rodada_real:
    print(f"Dados desatualizados! Iniciando coleta para sincronizar com a rodada {ultima_rodada_real}...")
    
    session = requests.Session()
    session.headers.update({'user-agent': 'Mozilla/5.0'})
    
    # Criamos uma nova lista para salvar o histórico atualizado
    # Você pode optar por estender a lista atual ou criar uma nova do zero
    novo_historico_completo = []

    for i, jogador in enumerate(data['elements']):
        p_id = jogador['id']
        url = f'https://fantasy.premierleague.com/api/element-summary/{p_id}/'
        
        try:
            r = session.get(url)
            if r.status_code == 200:
                dados_jogador = r.json().get('history', [])
                novo_historico_completo.extend(dados_jogador)
            
            time.sleep(0.2) 
            
            if i % 50 == 0:
                print(f"Processados {i} de {len(data['elements'])} jogadores...")
                
        except Exception as e:
            print(f"Erro no jogador {p_id}: {e}")
            continue

    # Salvando o novo arquivo

    # Nome do arquivo que será criado
    agora = datetime.now()
    data_formatada = agora.strftime("%d-%m-%Y_%H-%M-%S")
    nome_arquivo = f"historico_fpl_{data_formatada}.json"
    with open(nome_arquivo, 'w', encoding='utf-8') as f:
        json.dump(novo_historico_completo, f, indent=4, ensure_ascii=False)
        
    print(f"Sincronização concluída. Arquivo '{nome_arquivo}' gerado.")
else:
    print("Seus dados já estão atualizados com a última rodada finalizada. Nenhuma ação necessária.")

Última rodada finalizada na API: 32
Maior rodada no seu arquivo local: 32
Seus dados já estão atualizados com a última rodada finalizada. Nenhuma ação necessária.


# Explorando df de jogadores

In [20]:
# Criando o DataFrame dos jogadores

df_jogadores = pd.DataFrame(data['elements'])

In [21]:
for item in df_jogadores.columns.tolist():
    print(item)

can_transact
can_select
chance_of_playing_next_round
chance_of_playing_this_round
code
cost_change_event
cost_change_event_fall
cost_change_start
cost_change_start_fall
price_change_percent
dreamteam_count
element_type
ep_next
ep_this
event_points
first_name
form
id
in_dreamteam
news
news_added
now_cost
photo
points_per_game
removed
second_name
selected_by_percent
special
squad_number
status
team
team_code
total_points
transfers_in
transfers_in_event
transfers_out
transfers_out_event
value_form
value_season
web_name
known_name
region
team_join_date
birth_date
has_temporary_code
opta_code
minutes
goals_scored
assists
clean_sheets
goals_conceded
own_goals
penalties_saved
penalties_missed
yellow_cards
red_cards
saves
bonus
bps
influence
creativity
threat
ict_index
clearances_blocks_interceptions
recoveries
tackles
defensive_contribution
starts
expected_goals
expected_assists
expected_goal_involvements
expected_goals_conceded
corners_and_indirect_freekicks_order
corners_and_indirect_freeki

In [22]:
df_jogadores

,can_transact,can_select,chance_of_playing_next_round,chance_of_playing_this_round,code,cost_change_event,cost_change_event_fall,cost_change_start,cost_change_start_fall,price_change_percent,...,now_cost_rank_type,form_rank,form_rank_type,points_per_game_rank,points_per_game_rank_type,selected_rank,selected_rank_type,starts_per_90,clean_sheets_per_90,defensive_contribution_per_90
0,True,True,NaN,NaN,154561,0,0,5,-5,0,...,1,265,22,56,6,8,1,1.00,0.47,0.00
1,True,True,NaN,NaN,109745,0,0,-5,5,0,...,43,416,56,581,66,295,37,0.00,0.00,0.00
2,True,False,0.0,0.0,463748,0,0,0,0,0,...,53,435,66,598,75,376,53,0.00,0.00,0.00
3,True,True,NaN,NaN,551221,0,0,-1,1,0,...,89,403,49,570,60,368,52,0.00,0.00,0.00
4,True,True,100.0,100.0,226597,0,0,12,-12,0,...,1,89,30,2,1,5,1,1.00,0.56,9.26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
821,True,False,0.0,0.0,514362,0,0,0,0,0,...,272,618,223,717,289,806,358,0.00,0.00,0.00
822,True,True,NaN,NaN,589121,0,0,0,0,0,...,300,649,249,741,309,757,322,0.00,0.00,0.00
823,True,True,NaN,NaN,685628,0,0,0,0,0,...,322,676,270,763,326,747,316,0.00,0.00,0.00
824,True,True,100.0,100.0,209041,0,0,-3,3,0,...,239,227,102,423,185,723,300,1.32,0.26,6.58


# Explorando df de histórico

In [23]:
# Criando o DataFrame do histórico

df_historico = pd.DataFrame(data_hist)

In [24]:
for item in df_historico.columns.tolist():
    print(item)

element
fixture
opponent_team
total_points
was_home
kickoff_time
team_h_score
team_a_score
round
modified
minutes
goals_scored
assists
clean_sheets
goals_conceded
own_goals
penalties_saved
penalties_missed
yellow_cards
red_cards
saves
bonus
bps
influence
creativity
threat
ict_index
clearances_blocks_interceptions
recoveries
tackles
defensive_contribution
starts
expected_goals
expected_assists
expected_goal_involvements
expected_goals_conceded
value
transfers_balance
selected
transfers_in
transfers_out


In [25]:
df_historico

,element,fixture,opponent_team,total_points,was_home,kickoff_time,team_h_score,team_a_score,round,modified,...,starts,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,value,transfers_balance,selected,transfers_in,transfers_out
0,1,9,14,10,False,2025-08-17T15:30:00Z,0,1,1,False,...,1,0.00,0.00,0.00,1.52,55,0,1531911,0,0
1,1,11,11,6,True,2025-08-23T16:30:00Z,5,0,2,False,...,1,0.00,0.00,0.00,0.17,55,218659,2284634,277339,58680
2,1,25,12,2,False,2025-08-31T15:30:00Z,1,0,3,False,...,1,0.00,0.02,0.02,0.52,55,-12311,2406964,146739,159050
3,1,31,16,6,True,2025-09-13T11:30:00Z,3,0,4,False,...,1,0.00,0.00,0.00,0.20,55,171289,2765759,289041,117752
4,1,41,13,2,True,2025-09-21T15:30:00Z,1,1,5,False,...,1,0.00,0.01,0.01,0.89,55,-9786,2762632,98100,107886
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24650,817,264,8,2,False,2026-02-22T14:00:00Z,1,0,27,False,...,1,0.15,0.20,0.35,0.89,55,-2537,9723,748,3285
24651,817,280,2,4,True,2026-02-27T20:00:00Z,2,0,28,False,...,1,0.00,0.01,0.01,0.82,54,-1505,8326,560,2065
24652,817,290,12,1,True,2026-03-03T20:15:00Z,2,1,29,False,...,1,0.00,0.00,0.00,1.17,54,-653,7723,1109,1762
24653,817,292,5,5,False,2026-03-16T20:00:00Z,2,2,30,False,...,1,0.47,0.04,0.51,2.49,53,-812,7054,987,1799
